In [29]:
import pandas as pd
import os

from crewai import Agent, Task, Crew, Process, LLM, Flow
from crewai.flow.flow import listen, start
from crewai_tools import TavilySearchTool, ScrapeWebsiteTool
from pydantic import BaseModel, Field
from typing import List, Optional
from datetime import date

In [30]:
from dotenv import load_dotenv
import os

# Load .env.local from project root
# Notebook is at: backend/crews/sneaker_market_reseach/notebooks/
# Project root is 4 levels up: ../../../../.env.local
load_dotenv("../../../../.env.local", override=True)

TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
OPENAI_API_KEY = os.getenv("OPEN_AI_API_KEY")

In [31]:
search_tool = TavilySearchTool(api_key=TAVILY_API_KEY)
scrape_tool = ScrapeWebsiteTool()  
llm = LLM(model="gpt-4o-mini", api_key=OPENAI_API_KEY)

In [32]:
class ReleaseCandidate(BaseModel):
    product_name: str = Field(..., description="Name of the product")
    brand: Optional[str] = Field(None, description="Brand if known")
    release_date: date = Field(..., description="Date product releases")
    retail_price: Optional[int] = Field(..., description="Expected retail price of product")
    retailers: Optional[List[str]] = Field(..., description="Retailers confirmed to be selling the item")
    seed_sources: List[str] = Field(..., description="URLs confirming the release")

class ScoutOutput(BaseModel):
    candidates: List[ReleaseCandidate]

class ReleaseItems(ReleaseCandidate):
    resale_estimate: int = Field(..., description="Estimated resale value")
    confidence_score: float = Field(..., ge=0, le=100, description="Level of confidence from 0-100 that resale_estimate is correct")

class AnalystOutput(BaseModel):
    items: List[ReleaseItems]

## Agents

In [33]:
sneaker_scout = Agent(
    role="Upcoming Sneaker Release Scout",
    goal="""Identify upcoming sneaker releases.""",
    backstory="""
    You are a master web scraper who is highly resourceful and can easily navigate the internet to find relevant information and 
    Extract it in easily ingestible formats. You do not get distracted by irrelevant articles/information.
    """,
    tools=[search_tool, scrape_tool],
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

sneaker_market_analyst = Agent(
    role="Sneaker Resell Market Analyst",
    goal="Accurately project resale values for upcoming sneaker releases",
    backstory="""You are a long-time sneaker reseller and hypebeast. You have an expert understanding of how cultural trends,
    historical performance and market factors impact the potential profitability of sneakers on the secondary market.""",
    tools=[search_tool, scrape_tool],
    llm=llm,
    verbose=True,
    allow_delegation=False,
)


## Tasks

In [34]:
sneaker_scout_task = Task(
    description="""
    Compile information on upcoming sneaker releases by scraping information from reputable release calendars on
    sites such as https://www.sneakerfiles.com/release-dates/, nicekicks.com, sneakernews.com, and goat.com. Find and navigate to the release calendar pages on each site
    to find organized information on upcoming releases.
    
    Today is {today}. Only include releases between {today} and {cutoff_date}.

    Deliverable: Return up to {num_items} releases ordered by the soonest upcoming release.
    """,
    expected_output="""ScoutOutput with exactly {num_items} upcoming sneaker releases in the date window, each with 1–2 sources. You're outputted
    items should closely match the items on the release calendars of the sites.""",
    output_pydantic=ScoutOutput,
    agent=sneaker_scout,
)

sneaker_market_analyst_task = Task(
    description="""
    Analyze all sneaker releases given and predict resale price + confidence (0-100).

    Research similar past releases on StockX. Use historical prices of same model/line.
    Don't use pre-release prices (inflated).

    Confidence: 75-100=high (clear trend, ±$20), 50-75=moderate, 25-50=uncertain, 0-25=unique/no data.

    Limit: Max 2 searches per item. Use your knowledge when data is unavailable.

    Output: AnalystOutput with resale_estimate and confidence_score for each item.
    """,
    expected_output="""
    AnalystOutput with predictions for all items.
    """,
    output_pydantic=AnalystOutput,
    agent=sneaker_market_analyst,
    context=[sneaker_scout_task],
)

In [35]:
from datetime import date, timedelta

today = date.today()
cutoff = today + timedelta(days=21)
window_month = date.today()

# Change number of items
num_items = 3 

test_crew = Crew(
    agents=[sneaker_scout, sneaker_market_analyst],
    tasks=[sneaker_scout_task, sneaker_market_analyst_task],
    verbose=False,
    process=Process.sequential,
)

result = test_crew.kickoff(
    inputs={
        "today": today.isoformat(),
        "cutoff_date": cutoff.isoformat(),
        "num_items": num_items
    }
)

print(result)


╭────────────────────────────── 🤖 Agent Started ──────────────────────────────╮
│                                                                              │
│  Agent: Upcoming Sneaker Release Scout                                       │
│                                                                              │
│  Task:                                                                       │
│      Compile information on upcoming sneaker releases by scraping            │
│  information from reputable release calendars on                             │
│      sites such as https://www.sneakerfiles.com/release-dates/,              │
│  nicekicks.com, sneakernews.com, and goat.com. Find and navigate to the      │
│  release calendar pages on each site                                         │
│      to find organized information on upcoming releases.                     │
│                                                                              │
│      Today is 2026-01-13. O

╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): 



╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                                                              │
│  Info: Tracing has been disabled.                                            │
│                                                                              │
│  Your preference has been saved. Future Crew/Flow executions will not        │
│  collect traces.                                                             │
│                                                                              │
│  To enable tracing later, do any one of these:                               │
│  • Set tracing=True in your Crew/Flow code                                   │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file               │
│  • Run: crewai traces enable                                                 │
│                                                                              │
╰─────────────────────────

In [36]:
# Convert result to DataFrame
def result_to_dataframe(result):
    
    # Extract the AnalystOutput from result
    analyst_output = None
    
    # Try to get pydantic output
    if hasattr(result, 'pydantic'):
        analyst_output = result.pydantic
    elif hasattr(result, 'raw') and hasattr(result.raw, 'tasks_output'):
        if result.raw.tasks_output:
            last_task = result.raw.tasks_output[-1]
            analyst_output = last_task.pydantic if hasattr(last_task, 'pydantic') else None
    
    if not analyst_output:
        print("Warning: Could not extract AnalystOutput from result")
        return pd.DataFrame()
    
    if not hasattr(analyst_output, 'items'):
        print("Warning: AnalystOutput does not have 'items' attribute")
        return pd.DataFrame()
    
    # Convert items to list of dictionaries
    rows = []
    for item in analyst_output.items:
        # Handle list fields (retailers, seed_sources)
        retailers = item.retailers if item.retailers else []
        retailers_str = ", ".join(retailers) if isinstance(retailers, list) else str(retailers) if retailers else ""
        
        seed_sources = item.seed_sources if item.seed_sources else []
        seed_sources_str = ", ".join(seed_sources) if isinstance(seed_sources, list) else str(seed_sources) if seed_sources else ""
        
        row = {
            "Product Name": item.product_name,
            "Brand": item.brand if item.brand else "",
            "Release Date": item.release_date,
            "Retail Price": f"${item.retail_price}" if item.retail_price else "N/A",
            "Resale Estimate": f"${item.resale_estimate}" if item.resale_estimate else "N/A",
            "Confidence Score": f"{item.confidence_score:.1f}%" if item.confidence_score is not None else "N/A",
            "Retailers": retailers_str,
            "Sources": seed_sources_str
        }
        rows.append(row)
    
    df = pd.DataFrame(rows)
    return df

# Convert the result to a DataFrame
releases_df = result_to_dataframe(result)
display(releases_df)

,Product Name,Brand,Release Date,Retail Price,Resale Estimate,Confidence Score,Retailers,Sources
0,"Air Jordan 4 ""Flight Club""",Air Jordan,2026-01-17,$220,$212,80.0%,,https://www.sneakerfiles.com/air-jordan-4-flig...
1,"Nike Ja 3 ""Year of the Horse""",Nike,2026-01-16,$145,$170,75.0%,,https://www.sneakerfiles.com/nike-ja-3-year-of...
2,"Nike Air Force 1 Low ""Valentine's Day"" (Univer...",Nike,2026-01-14,$125,$80,65.0%,,https://www.sneakerfiles.com/nike-air-force-1-...


In [37]:


costs = 0.150 * (test_crew.usage_metrics.prompt_tokens + test_crew.usage_metrics.completion_tokens) / 1_000_000
print(f"Total costs: ${costs:.4f}")

# Convert UsageMetrics instance to a DataFrame
df_usage_metrics = pd.DataFrame([test_crew.usage_metrics.dict()])
df_usage_metrics

Total costs: $0.0085


,total_tokens,prompt_tokens,cached_prompt_tokens,completion_tokens,successful_requests
0,56986,49166,0,7820,14
